In [1]:
library(tidyverse)
library(data.table)
library(plotly)
library(glmnet)

Warning message:
“package ‘ggplot2’ was built under R version 4.4.3”
Warning message:
“package ‘tibble’ was built under R version 4.4.3”
── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.4     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Warning message:
“package ‘data.table’ was built under R version 4.4.3”

Attaching package: ‘data.table’


The following objects are masked from ‘package:lubridate’:

    hour, isoweek, mday, minute, month, quarter, second, wday, week,
    yday, year


The following objects are masked from ‘package:dplyr’:

    between, first, last


The

In [2]:
process_df <- function(df) {
    df <- fread(df)
    df$GO_ID <- as.factor(df$GO_ID)
    df <- df %>% pivot_longer(cols = c("div_d_gamb", "div_d_colu", "div_d_arab"), names_to = "Species", values_to = "Polymorphism")
    df$Species <- as.factor(df$Species)
    df <- df[!is.na(df$Polymorphism) & !is.infinite(df$Polymorphism) & df$Polymorphism > 0,]
    df$Level <- as.factor(df$Level)
    return(df)
}

lambda_cv <- function(gomatrix, df, alpha=1) {
    #df_matrix <- model.matrix(~GO_ID, data=df)

    divergence_vector <- df$residuals
    lambda_seq <- 10^seq(3, -3, by=-0.1)

    cv_output <- cv.glmnet(gomatrix, divergence_vector, alpha=alpha, lambda=lambda_seq, nfolds=10)
    best_lambda <- cv_output$lambda.min
    return(best_lambda)
}

lasso_regress <- function(gomatrix, df, alpha, lambda){
    #df_matrix <- model.matrix(~GO_ID, data=df)
    divergence_vector <- df$residuals

    lasso_model <- glmnet(gomatrix, divergence_vector, alpha=alpha, lambda=lambda)
    coefficients <- as.data.frame(as.matrix(coef(lasso_model)))
    colnames(coefficients) <- "Coefficient"
    coefficients <- rownames_to_column(coefficients, var="GO_ID")
    non_zero_coefficients <- coefficients[coefficients$Coefficient != 0,]
    return(non_zero_coefficients)
}

In [3]:
fdf <- process_df('out/gamb_colu_arab_Function_terms_stats.csv')
fdfG <- fdf[fdf$Species == "div_d_gamb",]
fdfA <- fdf[fdf$Species == "div_d_arab",]
fdfC <- fdf[fdf$Species == "div_d_colu",]
covar <- fread('out/gamb/gene_covariates.txt')
fdfG <- merge(covar, fdfG)
fdfG$Sexchrom <- FALSE
fdfG[fdfG$Chromosome == "AgamP4_X",]$Sexchrom <- TRUE
nrow(fdfG)

[1] 27230

In [4]:
compmat <- fread('out/gamb_colu_arab_Function_edge_matrix.txt')
compmat <- t(as.matrix(compmat))
compmat <- as.data.frame(compmat)
names(compmat) <- as.character(unlist(compmat[1,]))
compmat <- compmat[-1,]
compmat <- tibble::rownames_to_column(compmat, var="Gene")

In [5]:
mdf <- merge(fdfG, compmat)

In [6]:
resmodel <- lm(Polymorphism ~ GC_Content + Sexchrom + Length, data=mdf)
mdf$residuals <- resmodel$residuals


In [ ]:
L <- lambda_cv(as.matrix(select(mdf, contains("GO:"))), mdf, alpha=1)
print(L)
coefs <- lasso_regress(as.matrix(select(mdf, contains("GO:"))), mdf, alpha=1, lambda=L)
coefs %>% arrange(desc(Coefficient))

In [ ]:
cdf <- process_df('out/gamb_colu_arab_Component_terms_stats.csv')
cdfG <- cdf[cdf$Species == "div_d_gamb",]
cdfA <- cdf[cdf$Species == "div_d_arab",]
cdfC <- cdf[cdf$Species == "div_d_colu",]
covar <- fread('out/gamb/gene_covariates.txt')
cdfG <- merge(covar, cdfG)
cdfG$Sexchrom <- FALSE
cdfG[cdfG$Chromosome == "AgamP4_X",]$Sexchrom <- TRUE
head(cdfG)

Gene,Chromosome,Start,End,Length,GC_Content,GO_ID,Transcript,gene_pi_gamb,gene_pi_colu,⋯,ss_d_colu,ss_d_arab,ns_d_gamb,ns_d_colu,ns_d_arab,Level,Depth,Species,Polymorphism,Sexchrom
<chr>,<chr>,<int>,<int>,<int>,<dbl>,<fct>,<chr>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>,<int>,<fct>,<dbl>,<lgl>
AGAP000005,AgamP4_X,32382,38843,6462,0.42108,GO:0005694,AGAP000005-RA,0.003533088,0.001775797,⋯,-1.516408,-2.408548,-2.744734,-2.364059,-2.528620,5,5,div_d_gamb,1.034682,TRUE
AGAP000005,AgamP4_X,32382,38843,6462,0.42108,GO:0005634,AGAP000005-RA,0.003533088,0.001775797,⋯,-1.516408,-2.408548,-2.744734,-2.364059,-2.528620,5,5,div_d_gamb,1.034682,TRUE
AGAP000007,AgamP4_X,83817,88773,4957,0.57938,GO:0005886,AGAP000007-RA,0.003425350,0.001166049,⋯,-2.575448,-2.347921,-2.774236,-2.750270,-2.541279,3,3,div_d_gamb,1.048937,TRUE
AGAP000008,AgamP4_X,90142,94903,4762,0.47249,GO:0016020,AGAP000008-RA,0.003825916,0.003345175,⋯,-1.992269,-1.728950,-2.731823,-2.399738,-2.145399,2,2,div_d_gamb,1.044263,TRUE
AGAP000013,AgamP4_X,148957,163951,14995,0.48149,GO:0005886,AGAP000013-RA,0.009004225,0.007944258,⋯,-1.907544,-2.250249,-2.725971,-2.352487,-2.594408,3,3,div_d_gamb,1.067929,TRUE
AGAP000013,AgamP4_X,148957,163951,14995,0.48149,GO:0016020,AGAP000013-RA,0.009004225,0.007944258,⋯,-1.907544,-2.250249,-2.725971,-2.352487,-2.594408,2,2,div_d_gamb,1.067929,TRUE


In [ ]:
compmat <- fread('out/gamb_colu_arab_Component_edge_matrix.txt')
compmat <- t(as.matrix(compmat))
compmat <- as.data.frame(compmat)
names(compmat) <- as.character(unlist(compmat[1,]))
compmat <- compmat[-1,]
compmat <- tibble::rownames_to_column(compmat, var="Gene")

In [ ]:
mdf <- merge(cdfG, compmat)

In [ ]:
resmodel <- lm(Polymorphism ~ GC_Content + Sexchrom + Length, data=mdf)
mdf$residuals <- resmodel$residuals


In [ ]:
L <- lambda_cv(as.matrix(select(mdf, contains("GO:"))), mdf, alpha=1)
print(L)
coefs <- lasso_regress(as.matrix(select(mdf, contains("GO:"))), mdf, alpha=1, lambda=L)
coefs %>% arrange(desc(Coefficient))

[1] 0.001


GO_ID,Coefficient
<chr>,<dbl>
GO:0099023;GO:0017119,0.6673984
GO:0000502;GO:0031595,0.6634245
GO:0032045;GO:0071986,0.5967441
GO:1990234;GO:0000408,0.5789107
GO:0106068;GO:0030915,0.5555424
GO:0030684;GO:0030686,0.4866244
GO:1990234;GO:0106068,0.4505612
GO:0098796;GO:0000506,0.4209724
GO:0043227;GO:0065010,0.3793065


In [ ]:
pdf <- process_df('out/gamb_colu_arab_Process_terms_stats.csv')
pdfG <- pdf[pdf$Species == "div_d_gamb",]
pdfA <- pdf[pdf$Species == "div_d_arab",]
pdfC <- pdf[pdf$Species == "div_d_colu",]
covar <- fread('out/gamb/gene_covariates.txt')
pdfG <- merge(covar, pdfG)
pdfG$Sexchrom <- FALSE
pdfG[pdfG$Chromosome == "AgamP4_X",]$Sexchrom <- TRUE
head(pdfG)

Gene,Chromosome,Start,End,Length,GC_Content,GO_ID,Transcript,gene_pi_gamb,gene_pi_colu,⋯,ss_d_colu,ss_d_arab,ns_d_gamb,ns_d_colu,ns_d_arab,Level,Depth,Species,Polymorphism,Sexchrom
<chr>,<chr>,<int>,<int>,<int>,<dbl>,<fct>,<chr>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>,<int>,<fct>,<dbl>,<lgl>
AGAP000007,AgamP4_X,83817,88773,4957,0.57938,GO:0007411,AGAP000007-RA,0.003425350,0.001166049,⋯,-2.575448,-2.347921,-2.774236,-2.750270,-2.541279,8,8,div_d_gamb,1.048937,TRUE
AGAP000007,AgamP4_X,83817,88773,4957,0.57938,GO:0007155,AGAP000007-RA,0.003425350,0.001166049,⋯,-2.575448,-2.347921,-2.774236,-2.750270,-2.541279,2,2,div_d_gamb,1.048937,TRUE
AGAP000008,AgamP4_X,90142,94903,4762,0.47249,GO:0006457,AGAP000008-RA,0.003825916,0.003345175,⋯,-1.992269,-1.728950,-2.731823,-2.399738,-2.145399,2,2,div_d_gamb,1.044263,TRUE
AGAP000008,AgamP4_X,90142,94903,4762,0.47249,GO:0009408,AGAP000008-RA,0.003825916,0.003345175,⋯,-1.992269,-1.728950,-2.731823,-2.399738,-2.145399,3,4,div_d_gamb,1.044263,TRUE
AGAP000011,AgamP4_X,126414,134756,8343,0.51372,GO:0009081,AGAP000011-RA,0.006897925,0.006560240,⋯,-1.679237,-1.597132,-2.747789,-2.465054,-2.101571,5,7,div_d_gamb,1.131270,TRUE
AGAP000012,AgamP4_X,146181,147832,1652,0.56356,GO:0006418,AGAP000012-RA,0.009494305,0.005786689,⋯,-1.894390,-1.338811,-2.678551,-2.312434,-2.448316,7,9,div_d_gamb,1.118706,TRUE


In [ ]:
compmat <- fread('out/gamb_colu_arab_Process_edge_matrix.txt')
compmat <- t(as.matrix(compmat))
compmat <- as.data.frame(compmat)
names(compmat) <- as.character(unlist(compmat[1,]))
compmat <- compmat[-1,]
compmat <- tibble::rownames_to_column(compmat, var="Gene")

In [ ]:
mdf <- merge(pdfG, compmat)

In [ ]:
resmodel <- lm(Polymorphism ~ GC_Content + Sexchrom + Length, data=mdf)
mdf$residuals <- resmodel$residuals


In [ ]:
L <- lambda_cv(as.matrix(select(mdf, contains("GO:"))), mdf, alpha=1)
print(L)
coefs <- lasso_regress(as.matrix(select(mdf, contains("GO:"))), mdf, alpha=1, lambda=L)
coefs %>% arrange(desc(Coefficient))

[1] 0.001


GO_ID,Coefficient
<chr>,<dbl>
GO:0016925;GO:1990466,1.1303709
GO:0007114;GO:0007120,1.0883579
GO:0001510;GO:0036265,0.8396406
GO:0051604;GO:0016255,0.7007985
GO:0001919;GO:0001920,0.5485378
GO:0001101;GO:0043200,0.4216091
GO:0006081;GO:0046487,0.4115358
GO:0015850;GO:0015918,0.4061660
GO:0000740;GO:0007086,0.4025633
